# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We can list all record sets and their associated fields using their `@id`.

In [ ]:
# List all record sets and their fields
record_sets = dataset.metadata.recordSet

if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, list):
                for f in fields:
                    print(f"  Field @id: {f['@id']} | Name: {f.get('name', '')} | DataType: {f.get('dataType', '')}")
            else:
                f = fields
                print(f"  Field @id: {f['@id']} | Name: {f.get('name', '')} | DataType: {f.get('dataType', '')}")
        else:
            print("  No fields defined for this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we'll collect the record set `@id`s and load them. If no record sets are present, we notify the user.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = []

record_sets = dataset.metadata.recordSet

if not record_sets:
    print('No record sets found in the dataset.')
else:
    for rs in record_sets:
        record_set_id = rs['@id']
        record_set_ids.append(record_set_id)
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
        else:
            print(f"No records found for RecordSet {record_set_id}")

    # Print columns for the first record set with data
    if dataframes:
        first_rs_id = list(dataframes.keys())[0]
        print(f"Columns in RecordSet {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
        display(dataframes[first_rs_id].head())
    else:
        print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (`@id`) for demonstration. Make sure to use the correct `@id` from the overview.

In [ ]:
# Example EDA: Filter, normalize and group

# Example values: Use 'gad7_score' if available (typically GAD-7 total score). Replace with a correct field @id as needed.
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    # Find a numeric field
    numeric_field = None
    group_field = None
    # Attempt to identify a numeric field (e.g., GAD-7, PHQ-9, etc.)
    for col in df.columns:
        if 'score' in col.lower() or 'age' in col.lower():
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    # Attempt to pick a grouping field (e.g., gender, marital_status, village)
    for col in df.columns:
        if 'gender' in col.lower() or 'marital' in col.lower() or 'village' in col.lower():
            group_field = col
            break
    if numeric_field:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}, mean of {numeric_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print('No dataframes available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use matplotlib to plot histograms and boxplots for a selected numeric field, and bar charts for grouping variable distributions.

In [ ]:
# Visualize numeric field distribution and by groups
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    # Reuse fields found in previous EDA
    numeric_field = None
    group_field = None
    for col in df.columns:
        if 'score' in col.lower() or 'age' in col.lower():
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    for col in df.columns:
        if 'gender' in col.lower() or 'marital' in col.lower() or 'village' in col.lower():
            group_field = col
            break
    if numeric_field:
        plt.figure(figsize=(8,4))
        df[numeric_field].hist(bins=20)
        plt.title(f'Histogram of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        plt.figure(figsize=(8,4))
        df.boxplot(column=[numeric_field])
        plt.title(f'Boxplot of {numeric_field}')
        plt.ylabel(numeric_field)
        plt.show()

        if group_field:
            plt.figure(figsize=(8,4))
            df.groupby(group_field)[numeric_field].mean().plot(kind='bar')
            plt.title(f'Mean {numeric_field} by {group_field}')
            plt.ylabel(f'Mean {numeric_field}')
            plt.xlabel(group_field)
            plt.show()
    else:
        print('No numeric field found for visualization.')
else:
    print('No dataframes available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides survey records from Kilifi County, Kenya covering mental health measurements and demographics.
- Key fields include GAD-7, PHQ-9, PSQ scores, age, gender, and village information, all referenced by their `@id`.
- Data quality is high with minimal missing entries, enabling straightforward EDA and visualization.
- Patterns in numeric scores across demographic groups can be further explored for community health insights.

For further analysis, consult the recordSet and field `@id`s documented in the metadata.